In [6]:
import pandas as pd 
from sqlalchemy import create_engine

In [7]:
engine = create_engine(
    "mssql+pyodbc://@localhost\\SQLEXPRESS/Restaurant_Case_study?"
    "driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)
conn = engine.connect()
print("Connected successfully")
conn.close()


Connected successfully


In [8]:
import pandas as pd

query = """
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
"""

tables_df = pd.read_sql(query, engine)
tables_df


,TABLE_NAME
0,sales
1,menu
2,members


In [10]:
pd.read_sql("SELECT DB_NAME()", engine)

,
0,Restaurant_Case_study


In [66]:
sales = pd.read_sql("SELECT * FROM sales", engine)
menu = pd.read_sql("SELECT * FROM menu", engine)
members = pd.read_sql("SELECT * FROM members", engine)
print(sales)
print("*")
print(menu)
print("**")
print(members)

   customer_id  order_date  product_id
0            A  2025-01-01           1
1            A  2025-01-01           2
2            A  2025-01-07           2
3            A  2025-01-10           3
4            A  2025-01-11           3
5            A  2025-01-11           3
6            B  2025-01-01           2
7            B  2025-01-02           2
8            B  2025-01-04           1
9            B  2025-01-11           1
10           B  2025-01-16           3
11           B  2025-02-01           3
12           C  2025-01-01           3
13           C  2025-01-01           3
14           C  2025-01-07           3
*
   product_id product_name  price
0           1      biryani     10
1           2       paneer     15
2           3        dosai     12
**
  customer_id   join_date
0           A  2025-01-07
1           B  2025-01-09


In [11]:
sales.head()

,customer_id,order_date,product_id
0,A,2025-01-01,1
1,A,2025-01-01,2
2,A,2025-01-07,2
3,A,2025-01-10,3
4,A,2025-01-11,3


# Restaurant Sales Case Study Questions

**Dataset**:  
- Table `sales` (customer_id, order_date, product_id)  
- Table `menu` (product_id, product_name, price)  
- Table `members` (customer_id, join_date)  


In [10]:
# 1. Total amount spent by each customer.
#merge-->group--?aggregate
df=sales.merge(menu,on="product_id",how="inner")   # merge
g=(
    df.groupby("customer_id",as_index=False)["price"].sum().rename(columns={"price": "total_spent"}))
                                                            
print(g)


  customer_id  total_spent
0           A           76
1           B           74
2           C           36


In [12]:
# 2. Number of distinct visit days per customer.
df=sales.groupby("customer_id",as_index=False)["order_date"].count()
df


,customer_id,order_date
0,A,6
1,B,6
2,C,3


In [13]:
# 3. First item purchased by each customer.
df=sales.merge(menu,on="product_id",how="inner")
first_date=df.groupby("customer_id")["order_date"].min()
first_items = df.merge(first_date,on=["customer_id", "order_date"],how="inner")
result=first_items[["customer_id","product_name","order_date"]]
print(result)


  customer_id product_name  order_date
0           A      biryani  2025-01-01
1           A       paneer  2025-01-01
2           B       paneer  2025-01-01
3           C        dosai  2025-01-01
4           C        dosai  2025-01-01


In [50]:
# 4. Most purchased item and count.
df=sales.merge(menu,on="product_id",how="inner")
item_count = (df.groupby("product_name").size().reset_index(name="purchase_count")
)
print(item_count)

  product_name  purchase_count
0      biryani               3
1        dosai               8
2       paneer               4


In [76]:
# 5. Most popular item per customer.
df=sales.merge(menu,on="product_id",how="inner")
item_count= (df.groupby(["customer_id","product_name"]).size().reset_index(name="purchase_count"))
max_count = (item_count.groupby("customer_id")["purchase_count"].max().reset_index(name="max_purchase_count"))
result=item_count.merge(max_count,on="customer_id",how="inner")
final = result[
    result["purchase_count"] == result["max_purchase_count"]
][["customer_id", "product_name", "purchase_count"]]
print(final)

  customer_id product_name  purchase_count
1           A        dosai               3
3           B      biryani               2
4           B        dosai               2
5           B       paneer               2
6           C        dosai               3


In [154]:
# 6. First item after becoming a member.
# df=sales.merge(members,on="customer_id",how="inner")
# df_after = df[df["order_date"]>=df["join_date"]]
# df_menu=df_after.merge(menu,on="product_id",how="inner")
# first_date=(
#     df_menu.groupby("customer_id")["order_date"].min().reset_index()
# )
# print(first_date)
df=sales.merge(members,on="customer_id",how="inner")
df_after = df[df["order_date"]>=df["join_date"]]
df_menu=df_after.merge(menu,on="product_id",how="inner")
first_date=(df_menu.groupby("customer_id")["order_date"].max().reset_index(name="first_order_date"))
result = df_menu.merge(
    first_date,left_on=["customer_id","order_date"],right_on=["customer_id","first_order_date"],how="inner"
)
final = result[["customer_id","product_name"]]
print(final)
 






  customer_id product_name
0           A        dosai
1           A        dosai
2           B        dosai


In [29]:
# 7. Last item before becoming a member.
df=sales.merge(members,on="customer_id",how="inner")
df_before = df[df["order_date"] < df["join_date"]]
df_menu=df_before.merge(menu,on="product_id",how="inner")
last_date=(df_menu.groupby("customer_id")["order_date"].max().reset_index(name="last_order_date"))
result = df_menu.merge(
    last_date,
    left_on=["customer_id", "order_date"],
    right_on=["customer_id", "last_order_date"],
    how="inner"
)
final = result[["customer_id","product_name"]]
print(final)




  customer_id product_name
0           A      biryani
1           A       paneer
2           B      biryani


In [58]:
# 8. Items and amount before becoming a member.
df=sales.merge(members,on="customer_id",how="inner")
df_before=df[df["order_date"] < df["join_date"]]
df_menu=df_before.merge(menu,on="product_id",how="inner")
result=(
    df_menu.groupby(["customer_id","product_name"])["price"].sum().reset_index(name="total_price")
)

result

,customer_id,product_name,total_price
0,A,biryani,10
1,A,paneer,15
2,B,biryani,10
3,B,paneer,30


In [65]:
# 9. Loyalty points: 2x for biryani, 1x for others.
df=sales.merge(menu,on="product_id",how="inner")
df["points"]=df.apply(lambda x : x["price"]*2 if x["product_name"]=="biryani" else x["price"], axis=1)
final=df.groupby("customer_id")["points"].sum().reset_index(name="total_points")
final

,customer_id,total_points
0,A,86
1,B,94
2,C,36


In [86]:
# 10. Points during first 7 days after joining
df=sales.merge(members,on="customer_id",how="inner")
df_7days=df[(df["order_date"]>= df["join_date"]) & (df["order_date"] <= df["join_date"] + pd.Timedelta(days=6)) ]
df_name=df_7days.merge(menu,on="product_id",how="inner")
df_name["points"]=df_name.apply(lambda x : x["price"] * 2 if x["product_name"]=="biryani" else x["price"] , axis =1 )
print(df_name)
print(df)
final=df_name.groupby("customer_id")["points"].sum().reset_index(name="Total_points")
print(final)

  customer_id  order_date  product_id   join_date product_name  price  points
0           A  2025-01-07           2  2025-01-07       paneer     15      15
1           A  2025-01-10           3  2025-01-07        dosai     12      12
2           A  2025-01-11           3  2025-01-07        dosai     12      12
3           A  2025-01-11           3  2025-01-07        dosai     12      12
4           B  2025-01-11           1  2025-01-09      biryani     10      20
   customer_id  order_date  product_id   join_date
0            A  2025-01-01           1  2025-01-07
1            A  2025-01-01           2  2025-01-07
2            A  2025-01-07           2  2025-01-07
3            A  2025-01-10           3  2025-01-07
4            A  2025-01-11           3  2025-01-07
5            A  2025-01-11           3  2025-01-07
6            B  2025-01-01           2  2025-01-09
7            B  2025-01-02           2  2025-01-09
8            B  2025-01-04           1  2025-01-09
9            B  2025-0

In [113]:
# 11. Total spent on biryani.
df=sales.merge(menu,on="product_id",how="inner")
#df_group=df.groupby("product_name")["price"].sum()
print(df)
print(df_group)
#df_total=df_group.loc["biryani"]
df_total = (df.groupby("product_name")["price"].sum().reset_index().query("product_name == 'biryani'")
)
print(df_total)


   customer_id  order_date  product_id product_name  price
0            A  2025-01-01           1      biryani     10
1            A  2025-01-01           2       paneer     15
2            A  2025-01-07           2       paneer     15
3            A  2025-01-10           3        dosai     12
4            A  2025-01-11           3        dosai     12
5            A  2025-01-11           3        dosai     12
6            B  2025-01-01           2       paneer     15
7            B  2025-01-02           2       paneer     15
8            B  2025-01-04           1      biryani     10
9            B  2025-01-11           1      biryani     10
10           B  2025-01-16           3        dosai     12
11           B  2025-02-01           3        dosai     12
12           C  2025-01-01           3        dosai     12
13           C  2025-01-01           3        dosai     12
14           C  2025-01-07           3        dosai     12
customer_id  product_name
A            biryani         1

In [132]:
# 12. Customer with most dosai orders.
df=sales.merge(menu,on="product_id",how="inner")
df_dosai = df[df["product_name"]=="dosai"]
dosai_count=df_dosai.groupby("customer_id").size().reset_index(name="dosai_orders")
max_order=dosai_count["dosai_orders"].max()
final=dosai_count[dosai_count["dosai_orders"]==max_orders]
print(final)

  customer_id  dosai_orders
0           A             3
2           C             3


In [150]:
# 13. Average spend per visit.
# ak visit=ak din m kitni bar like 2025-01-01 → biryani, dosai,2025-01-01 → paneer
df=sales.merge(menu,on="product_id",how="inner")
df_group=df.groupby(["customer_id","order_date"]).agg( visit_spend=("price", "sum") ).reset_index()
final=df_group.groupby("customer_id")["visit_spend"].mean().reset_index(name="avg_spend_per_visit")
print(final)

  customer_id  avg_spend_per_visit
0           A            19.000000
1           B            12.333333
2           C            18.000000
